# Least Squares Method
The least squares method is a statistical technique used to find the best-fitting curve or line through a set of data points. 

It minimizes the sum of the squares of the differences between the observed values and the values predicted by the model. 

This method is commonly used in regression analysis to estimate the parameters of a linear or nonlinear model.

Here is a simple example of how to implement the least squares method in Python.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px 


## Generate synthetic data
We create a simple linear relationship with some added noise to simulate real-world data.

In [2]:
# By setting a random seed, we ensure that the generated data is reproducible.
np.random.seed(13)
df= pd.DataFrame(data={'x': np.linspace(0,10,11), 'y': np.linspace(0,10,11)*2+ np.random.normal(3.0, 1.2, 11)})

df= df.round(3)

df.to_csv(path_or_buf="../Data/linear_relationship.csv", index=False)



## Read the csv file
extract the x values and y values from the csv file.

calculate the necessary summations for the least squares method.

calculate the optimal slope(m) and intercept(b) for the best-fitting line.

plot the synthetic data points and the best-fitting line.


In [3]:
df= pd.read_csv("../Data/linear_relationship.csv")
xs= df['x']
ys= df['y']
n= len(xs)
sum_x = np.sum(xs)
sum_y = np.sum(ys)
sum_xx = np.sum(xs**2)
sum_xy = np.sum(xs*ys)
sum_yy = np.sum(ys**2)
det_A = 4* ( n*sum_xx - sum_x**2 )
m_opt = 4* (n*sum_xy - sum_x * sum_y) / det_A 
b_opt = 4* (sum_xx * sum_y - sum_x * sum_xy) / det_A

def predict(x):
    return m_opt * x + b_opt

print(f"optimal slope m: {m_opt:.3f}")
print(f"optimal intercept b: {b_opt:.3f}")



optimal slope m: 1.976
optimal intercept b: 3.576


In [4]:
# 使用Plotly Express 生成了一个带边际分布图的回归散点图。
# 在主散点图的上方添加一个边际图，用于展示 x 轴数据的分布情况。
# 在主散点图的右侧添加一个边际图，用于展示 y 轴数据的分布情况。
# "violin" 表示小提琴图（Violin Plot），它是核密度估计（KDE）的镜像对称形式。
fig = px.scatter(df, x="x", y="y", marginal_y="violin",
           marginal_x="box", trendline="ols", template="simple_white")
fig.show()


In [5]:
fig_regression =go.Figure()
fig_regression.add_trace(go.Scatter(x=xs, y=ys, mode='markers', name='Original Data', 
                                    marker=dict(color='blue', size=4, symbol='circle')))

x_fit = np.linspace(min(xs), max(xs), 20)
y_fit = predict(x_fit)
fig_regression.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', 
                                    name= f'y_predict = {m_opt:.2f}x + {b_opt:.2f}', 
                                    line=dict(color='orange', width=2, dash='solid')))
# I need to customize the layout of the plot to make it more informative and visually appealing. This includes setting the spine styles, grid styles, legend position, tick parameters, and title properties.
fig_regression.update_layout(title=dict(text='Linear Regression using Least Squares Method', 
                                        x=0.5, y=0.8, font=dict(size=15, color='black', family='Arial')),
                             xaxis_title='x', yaxis_title='y', legend_title=None, 

                             legend=dict(x=0.02, y=0.98, bgcolor="#FDFDFD"), 

                             xaxis=dict(range=[min(xs), max(xs)], showline=True, mirror=True, showgrid=True, gridcolor="#F7F7F7", ticks='outside', tickwidth=1,
                             minor=dict(ticks='outside', showgrid=True, gridcolor="#FDFDFD", dtick=0.5)),

                             yaxis=dict(range=[0, 25], showline=True, mirror=True, showgrid=True, gridcolor="#F7F7F7", ticks='outside', tickwidth=1, minor=dict(ticks='outside', showgrid=True, gridcolor="#FBFBFB", dtick=1 )),

                             width=600, height=400, template='simple_white', title_x=0.5, title_font=dict(size=15, color='black', family='Arial'))
fig_regression.show()


## Generate coordinates for the least squares error function

Before plotting the 3D surface and contour plots, we need to generate a grid of slope (m) and intercept (b) values, and compute the least squares error for each combination of m and b.

In [6]:
def residuals_squared(xs, ys, m, b):
    y_pred = m * xs + b
    residuals= ys - y_pred
    return np.sum(residuals**2)

z_min = residuals_squared(xs, ys, m_opt, b_opt)

print(f"minimum sum of squared residuals: {z_min:.3f}")

m_range = 2
b_range = 2
m_values = np.linspace(m_opt - m_range, m_opt + m_range, 100)
b_values = np.linspace(b_opt - b_range, b_opt + b_range, 100)

M, B = np.meshgrid(m_values, b_values)
Z = sum_xx * M**2 + 2 * sum_x * M * B - 2* sum_xy * M + n * B**2 - 2 * sum_y * B + sum_yy


minimum sum of squared residuals: 11.816


import plotly.express as px
df = px.data.election()
fig = px.scatter_3d(df, x="Joly", y="Coderre", z="Bergeron", color="winner", size="total", hover_name="district",
                  symbol="result", color_discrete_map = {"Joly": "blue", "Bergeron": "green", "Coderre":"red"})
fig.show()

## 3D surface plot of the least squares error function

To visualize the least squares error function, we can create a 3D surface plot. 

This plot will show how the error changes with different values of the slope (m) and intercept (b).

The  optimal point (m_opt, b_opt) will be highlighted on the surface plot to show where the least squares error is minimized.

In [7]:
fig_surface = go.Figure()
fig_surface.add_trace(
    go.Surface(x=M, y=B, z=Z, colorscale='Teal', showscale=True, opacity=0.7, 
               name= 'Residuals_squared Surface', 
               colorbar= dict(title=dict(text='', side='top'), x=0.9, y=0.5, len=0.5, thickness=30)))

fig_surface.add_trace(
    go.Scatter3d(x=[m_opt,], y=[b_opt,], z=[z_min,], 
                    mode= 'markers+text', marker=dict(color='red', size=4, symbol='circle'), 
                    text=[f'Optimal point: <br> m={m_opt:.2f}<br> b={b_opt:.2f}<br> z(m,b)={z_min:.2f}'] , 
                    textposition='top center', 
                    textfont=dict(size=13, weight='bold', color="#B8B8B8", family='Arial'),
                    name='Optimal (m,b)'))

fig_surface.update_layout(
    title= dict(text='Residual Squared Surface for linear regression <br>  <sub>Red point: partial deratives = 0, Optimal (m,b) </sub>', x=0.5, y=0.8, 
                font=dict(size=18, color='black', family='Arial')),
    scene=dict(xaxis=dict(title=dict(text='Slope(m)', font=dict(size=18))),
               yaxis=dict(title=dict(text='Intercept(b)', font=dict(size=18))),
               zaxis=dict(title=dict(text='z(m,b)', font=dict(size=18))),
               # Camera position in normalized coordinates (relative to the 3D plot center)
               # x, y, z values are dimensionless and control the viewing angle and distance
               # x=2: camera distance along the Slope(m) axis
               # y=0: camera distance along the Intercept(b) axis  
               # z=1: camera distance along the Residuals Squared axis
               # Larger values move the camera farther away in that direction
               camera= dict(eye=dict(x=1.2, y=-2.2, z=0.5)),),
    width=800, height=800, 
    margin=dict(l=0, r=0, b=0, t=0),
    )
fig_surface.write_html("../docs/least_squares_residuals_surface.html")
fig_surface.write_image("../docs/images/least_squares_residuals_surface.svg")
fig_surface.show()


## Create a Contour plot of the least squares error function

This contour plot is used to visualize that the least squares error function has a minimun at the optimal point (m_opt, b_opt).

Closed eclipse-shape contours indicate that the error function is convex and has a single minimun, which is the optimal solution for the slope and intercept.

In [8]:
fig_contour =go.Figure()

fig_contour.add_trace(
    # coloring='heatmap': Use heatmap coloring to fill contour regions with continuous color gradient
    # This provides better visualization of the error function's topology
    # colorscale: color mapping scheme; 'Teal' uses a teal color gradient to represent different z-value ranges
    go.Contour(x=m_values, y=b_values, z= np.log(Z), colorscale='Teal', 
               contours=dict(showlabels=False, 
                             start= np.log(z_min), end=np.log(np.max(Z))+ 0.2, size=0.2,),
                name='Residuals_squared Contour',
                colorbar=dict(title=dict(text='z(m,b)', side='top'), 
                            x=0.95, y=0.45, len=0.9, thickness=50,
                            tickvals=np.log([z_min, 32.1, 87.4, 237.5, 645.5, np.max(Z)]),ticktext=[f'{v:.1f}' for v in [z_min, 32.1, 87.4, 237.5, 645.5, np.max(Z)]]
                            ) ) )
fig_contour.add_trace(
    go.Scatter(x=[m_opt,], y=[b_opt,],mode= 'markers + text', 
               marker=dict(color='red', size=8, symbol='star'),
               text=[f'Optimal point: <br> m={m_opt:.2f} <br> b={b_opt:.2f} <br> z(m,b)={z_min:.2f}'],
               textposition='bottom center',
               textfont=dict(size=15, weight='bold', color="#FFFFFF", family= 'Arial')) )

fig_contour.update_layout(
    title=dict(text='Contour Plot of Residuals Squared Function z(m,b) <br> <sub> Closed eclipse-shaped contours indicate a single minimum point</sub>',
               x= 0.45, y= 0.9, font=dict(size=20, color='black', family= 'Arial'),),
               # standoff: distance in pixels between title and axis line
               xaxis=dict(domain=[0, 0.9], title= dict(text='Slope(m)', font=dict(size=18), standoff=10)),
               yaxis=dict(domain=[0, 0.9], title= dict(text='Intercept(b)', font=dict(size=18), standoff=10)),
               width=800, height=800,
               margin=dict(l=50, r=50, b=50, t=50))

fig_contour.write_image("../docs/images/least_squares_residuals_contour.svg")

fig_contour.show()

